## load and prepare data

In [ ]:
%cd ../..
%matplotlib inline

import itertools
import os
import tempfile
import subprocess

import nibabel as nib
import numpy as np

import sys
sys.path.insert(0, 'evaluation_meldgraph')
from vol_eval_plots import (GROUP_3T, GROUP_7T_ADAPTED, GROUP_7T_DEFAULT, load_and_prepare_data,
                            filter_to_common_subjects, harmo_labels, plot_on_surface)


In [ ]:
eval_stats_df = load_and_prepare_data()

## plot FP distributions

In [ ]:
def sum_predictions(subj_dirs):
    with tempfile.TemporaryDirectory(dir='tmp/') as temp_dir:
        cmd = f'apptainer run freesurfer_8.1.0.sif /bin/bash -c "cp \$FREESURFER_HOME/subjects/fsaverage_sym/surf/lh.inflated {temp_dir}"'
        subprocess.run(cmd, shell=True, check=True)
        cmd = f'apptainer run freesurfer_8.1.0.sif /bin/bash -c "cp \$FREESURFER_HOME/subjects/fsaverage_sym/label/lh.cortex.label {temp_dir}"'
        subprocess.run(cmd, shell=True, check=True)
        fsaverage_surf = nib.freesurfer.io.read_geometry(f'{temp_dir}/lh.inflated')
        fsaverage_cortex = nib.freesurfer.io.read_label(f'{temp_dir}/lh.cortex.label')
        cortex_mask = np.zeros(fsaverage_surf[0].shape[0], dtype=bool)
        fsaverage_length = fsaverage_surf[0].shape[0]
        cortex_mask[fsaverage_cortex] = True


    sum_fps = {}
    sum_fps['lh'] = np.zeros(fsaverage_length)
    sum_fps['rh'] = np.zeros(fsaverage_length)

    sum_tps = {}
    sum_tps['lh'] = np.zeros(fsaverage_length)
    sum_tps['rh'] = np.zeros(fsaverage_length)

    sum_tp_gts = {}
    sum_tp_gts['lh'] = np.zeros(fsaverage_length)
    sum_tp_gts['rh'] = np.zeros(fsaverage_length)

    sum_fn_gts = {}
    sum_fn_gts['lh'] = np.zeros(fsaverage_length)
    sum_fn_gts['rh'] = np.zeros(fsaverage_length)

    for subj_dir in subj_dirs:
        for hemi in ['lh', 'rh']:
            print(subj_dir)
            path_prediction_labeled = f'{subj_dir}/{hemi}.prediction.mgh'
            if not os.path.exists(path_prediction_labeled):
                print(f'No prediction found for {subj_dir}, skipping...')
                continue
            prediction_labeled_data = nib.load(path_prediction_labeled)

            sum_fps[hemi] += prediction_labeled_data.get_fdata().squeeze() == 1
                # false-positives are labeled as 1 in the output of vol_eval.py
            sum_tps[hemi] += prediction_labeled_data.get_fdata().squeeze() == 2
                # true-positives are labeled as 2 in the output of vol_eval.py

            path_gt_labeled = f'{subj_dir}/{hemi}.gt_labeled.fsaverage_sym.mgh'
            if not os.path.exists(path_gt_labeled):
                print(f'No GT found for {subj_dir}, skipping GT-based metrics...')
                continue
            gt_labeled = nib.load(path_gt_labeled)

            sum_tp_gts[hemi] += gt_labeled.get_fdata().squeeze() == 2
                # true-positive GT vertices are labeled as 2 in the output of vol_eval.py
            sum_fn_gts[hemi] += gt_labeled.get_fdata().squeeze() == 1
                # false-negative GT vertices are labeled as 1 in the output of vol_eval.py

    # sum lh and rh
    sum_fps = sum_fps['lh'] + sum_fps['rh']
    sum_tps = sum_tps['lh'] + sum_tps['rh']
    sum_tp_gts = sum_tp_gts['lh'] + sum_tp_gts['rh']
    sum_fn_gts = sum_fn_gts['lh'] + sum_fn_gts['rh']

    sum_fps[cortex_mask == False] = np.nan
    sum_tps[cortex_mask == False] = np.nan
    sum_tp_gts[cortex_mask == False] = np.nan
    sum_fn_gts[cortex_mask == False] = np.nan

    denom_sens = sum_tp_gts + sum_fn_gts
    sensitivity = np.full_like(denom_sens, np.nan, dtype=float)
    np.divide(sum_tp_gts, denom_sens, out=sensitivity, where=denom_sens != 0)

    denom_ppv = sum_tps + sum_fps
    precision_ppv = np.full_like(denom_ppv, np.nan, dtype=float)
    np.divide(sum_tps, denom_ppv, out=precision_ppv, where=denom_ppv != 0)

    return sum_fps, sum_tps, sum_tp_gts, sum_fn_gts, sensitivity, precision_ppv

In [ ]:
# for the analysis groups 3T, 7T_default, 7T_adapted and the harmo conditions noharmo, harmo,
# plot the summed FP distributions on the fsaverage_sym surface

analysis_groups_fp = [GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED]
harmo_conditions_fp = ['noharmo', 'harmo']

# restrict to the same fixed set of subjects used for the performance tables above (present
# under every (harmo, analysis_group) combination), and to subjects with a successful reconstruction
# (path_labeled_pred_surfs/subject-ID only exists on disk in that case)
eval_stats_df_fp = eval_stats_df[(eval_stats_df['recon_successful'] == 1)] #& (eval_stats_df['group'] == 'control')] # optionally only plot controls
eval_stats_df_fp = filter_to_common_subjects(eval_stats_df_fp, analysis_groups_fp, harmo_conditions_fp)

sum_fps_by_condition = {}
for analysis_group, harmo in itertools.product(analysis_groups_fp, harmo_conditions_fp):
    df_condition = eval_stats_df_fp[(eval_stats_df_fp['harmo'] == harmo) & (eval_stats_df_fp['analysis_group'] == analysis_group)]

    subj_dirs = [
        f"{row['path_labeled_pred_surfs']}/{row['subject ID']}"
        for _, row in df_condition.iterrows()
    ]

    sum_fps, sum_tps, sum_tp_gts, sum_fn_gts, sensitivity, precision_ppv = sum_predictions(subj_dirs)
    sum_fps_by_condition[(analysis_group, harmo)] = (sum_fps, len(subj_dirs))

# shared color scale across all panels so the summed-FP maps are directly comparable
vmax_fp = max(np.nanmax(sum_fps) for sum_fps, _ in sum_fps_by_condition.values())

### Figure 4

In [ ]:
data_gt = []
labels_gt = []

# first add a map of the summed ground truths
data_gt.append(sum_tp_gts + sum_fn_gts)
labels_gt.append(f"Ground truth\n FCD labels")

p = plot_on_surface(data_gt, 
                labels_gt,
                cmap='Greens',
                gamma=0.5)

p.show()

In [ ]:
data = []
labels = []

for analysis_group, harmo in itertools.product(analysis_groups_fp, harmo_conditions_fp):
    sum_fps, n_subj = sum_fps_by_condition[(analysis_group, harmo)]
    data.append(sum_fps)
    labels.append(f"{analysis_group} {harmo_labels[harmo]}")
    print(n_subj, "subjects in", analysis_group, harmo, "condition")

p = plot_on_surface(data, 
                labels, 
                vmax=vmax_fp, 
                cmap='Reds',
                gamma=0.5)

p.show()